In [1]:
!pip -q install pandas requests openpyxl rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 55.4 MB/s eta 0:00:00


In [1]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
input_file = next(iter(uploaded))

if input_file.lower().endswith((".xlsx", ".xls")):
    excel = pd.ExcelFile(input_file)
    print("Available sheets:", excel.sheet_names)

    # Read the deduplicated compounds sheet
    sheet_name = "Unique_by_CAS"
    if sheet_name not in excel.sheet_names:
        raise ValueError(
            f"Sheet '{sheet_name}' not found. Available sheets: "
            f"{excel.sheet_names}"
        )

    df = pd.read_excel(input_file, sheet_name=sheet_name)
else:
    df = pd.read_csv(input_file)

print("Number of compounds:", len(df))
print("Columns:", df.columns.tolist())
display(df.head())

Saving compunds_unique_by_CAS.xlsx to compunds_unique_by_CAS.xlsx
Available sheets: ['Unique_by_CAS', 'Repeated_CAS_Rows', 'CAS_Name_Check', 'Missing_CAS', 'Summary']
Number of compounds: 74
Columns: ['Code', 'Trivial Name', 'CAS #', '5 µg/mL', '10 µg/mL', '20 µg/mL']


,Code,Trivial Name,CAS #,5 µg/mL,10 µg/mL,20 µg/mL
0,RM1,Irganox 1010,6683-19-8,0.557,0.537,0.521
1,RM2,Irganox 1076,2082-79-3,0.012,0.019,0.017
2,RM4,Butylated hydroxytoluene,128-37-0,0.320,0.390,0.435
3,RM5,"2,4-Di-t-butyl phenol",96-76-4,n.d.,n.d.,n.d.
4,RM8,Di-(2-ethylhexyl)phthalate,117-81-7,0.650,0.552,0.635


In [6]:
import sys
import subprocess

# Ensure rdkit is installed in the current environment
try:
    from rdkit import Chem
except ModuleNotFoundError:
    print("RDKit not found. Installing rdkit...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit"])
    from rdkit import Chem
    print("RDKit installed successfully!")

import requests
import pandas as pd
import time

def get_smiles_from_cas(cas_id):
    if pd.isna(cas_id) or not str(cas_id).strip():
        return None, "Missing CAS"

    cas_str = str(cas_id).strip()

    # Strategy 1: PubChem Name Endpoint (Query CAS as a name)
    pubchem_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/chemical/name/{cas_str}/property/CanonicalSMILES/JSON"
    try:
        response = requests.get(pubchem_url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            smiles = data["PropertyTable"]["Properties"][0]["CanonicalSMILES"]
            return smiles, "Success (PubChem)"
    except Exception:
        pass

    # Strategy 2: CACTUS (Chemical Identifier Resolver) fallback
    cactus_url = f"https://cactus.nci.nih.gov/chemical/structure/{cas_str}/smiles"
    try:
        response = requests.get(cactus_url, timeout=10)
        if response.status_code == 200 and response.text.strip():
            smiles = response.text.strip()
            if "<html>" not in smiles.lower():  # Check it's not an error page
                return smiles, "Success (CACTUS)"
    except Exception:
        pass

    return None, "Not Found"

print("Retrieving SMILES using PubChem & CACTUS fallbacks...")
results = []
for idx, row in df.iterrows():
    cas = row["CAS #"]
    smiles, status = get_smiles_from_cas(cas)

    # Validate SMILES using RDKit
    rdkit_valid = False
    if smiles:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            rdkit_valid = True

    results.append({
        "Code": row.get("Code"),
        "Trivial Name": row.get("Trivial Name"),
        "CAS #": cas,
        "SMILES": smiles,
        "RDKit_Valid": rdkit_valid,
        "Status": status
    })
    time.sleep(0.2)  # Respect API rate limits

smiles_df = pd.DataFrame(results)

# Compute and display statistics
total = len(smiles_df)
retrieved = smiles_df["SMILES"].notna() & (
    smiles_df["SMILES"].astype(str).str.strip() != ""
)
valid = smiles_df["RDKit_Valid"].fillna(False).astype(bool)

retrieval_rate = retrieved.sum() / total * 100 if total else 0
validity_rate = valid.sum() / retrieved.sum() * 100 if retrieved.sum() else 0

print("\n--- STATISTICS ---")
print("Total CAS numbers:", total)
print("SMILES retrieved:", int(retrieved.sum()))
print("RDKit-valid SMILES:", int(valid.sum()))
print(f"SMILES retrieval rate: {retrieval_rate:.2f}%")
print(f"RDKit validity rate among retrieved SMILES: {validity_rate:.2f}%")

Retrieving SMILES using PubChem & CACTUS fallbacks...


[05:55:24] SMILES Parse Error: syntax error while parsing: [Zn++](|[S-]c1cccc[n+]1[O-])|[S-]c2cccc[n+]2[O-]
[05:55:24] SMILES Parse Error: check for mistakes around position 8:
[05:55:24] [Zn++](|[S-]c1cccc[n+]1[O-])|[S-]c2cccc[n
[05:55:24] ~~~~~~~^
[05:55:24] SMILES Parse Error: Failed parsing SMILES '[Zn++](|[S-]c1cccc[n+]1[O-])|[S-]c2cccc[n+]2[O-]' for input: '[Zn++](|[S-]c1cccc[n+]1[O-])|[S-]c2cccc[n+]2[O-]'



--- STATISTICS ---
Total CAS numbers: 74
SMILES retrieved: 70
RDKit-valid SMILES: 69
SMILES retrieval rate: 94.59%
RDKit validity rate among retrieved SMILES: 98.57%


In [7]:
from google.colab import files

failed_df = smiles_df[
    ~smiles_df["Status"].eq("Success")
].copy()

summary_df = pd.DataFrame({
    "Metric": [
        "Total unique CAS numbers",
        "SMILES retrieved",
        "RDKit-valid SMILES",
        "SMILES retrieval rate (%)",
        "RDKit validity rate among retrieved (%)",
        "Compounds requiring review"
    ],
    "Value": [
        total,
        int(retrieved.sum()),
        int(valid.sum()),
        round(retrieval_rate, 2),
        round(validity_rate, 2),
        len(failed_df)
    ]
})

excel_output = "CLAP_CAS_to_SMILES_PubChem.xlsx"
csv_output = "CLAP_CAS_to_SMILES_PubChem.csv"

with pd.ExcelWriter(excel_output, engine="openpyxl") as writer:
    smiles_df.to_excel(writer, sheet_name="All_SMILES", index=False)
    failed_df.to_excel(writer, sheet_name="Needs_Review", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

smiles_df.to_csv(csv_output, index=False)

files.download(excel_output)
files.download(csv_output)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>